# FT-900 半导体出口原始发布来源预检

冻结范围：三个官方月报 PDF，读取原始字节并核对 NAS 收据；仅核查 Exhibit 7 出口行与报表印出的发布时间。无行情结果、模型或账户。

In [1]:
from pathlib import Path
import hashlib,json,re,fitz
root=Path.cwd();card=root/'research/census_ft900_semiconductor_export_source_20261023'
plan=json.loads((card/'plan.json').read_text());capture=json.loads((card/'capture_receipts.json').read_text());nas=json.loads((card/'nas_binding.json').read_text())
assert len(capture['receipts'])==plan['budget']['exact_PDF_GETs']==3 and nas['remote_hash_verified']
months={'ft900_1112.pdf':('2011-12','FEBRUARY 10, 2012'), 'ft900_1705.pdf':('2017-05','JULY 6, 2017'), 'ft900_2404.pdf':('2024-04','JUNE 6, 2024')}
rows=[]
for receipt in capture['receipts']:
 path=root/receipt['local_path'];payload=path.read_bytes();assert hashlib.sha256(payload).hexdigest()==receipt['sha256']==nas['entries'][path.name]
 doc=fitz.open(stream=payload,filetype='pdf');month,expected_release=months[path.name]
 first=' '.join(doc[0].get_text().split());assert expected_release in first and '8:30' in first
 pages=[]
 for page_number,page in enumerate(doc,1):
  text=page.get_text()
  if 'Exhibit 7.' in text and 'Exports of Goods by End-Use Category and Commodity' in text and re.search(r'(?m)^Semiconductors\s*$',text):pages.append((page_number,text))
 assert len(pages)==1,(path.name,len(pages))
 page_number,text=pages[0];lines=[x.strip() for x in text.splitlines()];idx=lines.index('Semiconductors')
 current,prior,reported_change=(int(lines[idx+j].replace(',','')) for j in (1,2,3))
 rows.append({'reference_month':month,'file':path.name,'pdf_sha256':receipt['sha256'],'pdf_bytes':len(payload),'printed_release_date':expected_release,'printed_release_time':'08:30 US Eastern as printed','exhibit':7,'pdf_page':page_number,'row':'Semiconductors','current_month_million_usd':current,'prior_month_in_same_release_million_usd':prior,'printed_change_million_usd':reported_change,'change_minus_rounded_arithmetic':reported_change-(current-prior)})
result={'status':'three_original_monthly_PDF_export_rows_present_source_preflight_only','plan_sha256':hashlib.sha256((card/'plan.json').read_bytes()).hexdigest(),'nas_archive_sha256':nas['archive_sha256'],'nas_remote_hash_verified':True,'samples':rows,'historical_first_public_bytes_qualified':False,'full_2011_2022_coverage_qualified':False,'classification_continuity_qualified':False,'model_input_qualified':False,'model_fits':0,'native_accounts':0,'market_outcome_reads':0,'SOXX_goal_achieved':False}
(card/'source_preflight_result.json').write_text(json.dumps(result,indent=2)+'\n')
print([(r['reference_month'],r['current_month_million_usd'],r['prior_month_in_same_release_million_usd'],r['printed_release_date']) for r in rows])

[('2011-12', 3438, 3503, 'FEBRUARY 10, 2012'), ('2017-05', 3862, 3865, 'JULY 6, 2017'), ('2024-04', 4856, 4492, 'JUNE 6, 2024')]


## 结论

三个抽样月份的原始月报均有 Exhibit 7 “Semiconductors”出口行，且报表印出美国东部 08:30 发布时间。当前下载到的 PDF 并不能单独证明其字节就是历史首次公开版本；2012 分类调整和后续年度修订也要求独立核查。来源预检通过行存在性，因子与 Stage1 信息检验均未获准，SOXX 目标未实现。

后续一次官方索引读取得到 143 个直读月份；2019-02 官方标题误拼 `Ferbruary 2019`，按单独冻结的 URL/分区/邻月合同核对后索引有 144 个训练月份标题，但逐链接核对仅 143 个指向相应月份；August 2015 误链到 September PDF。详见后续 `index_link_correction.json`，本卡未取得 2015-08 PDF 内容资格。URL 覆盖不等于 PDF 内容或历史首次公开字节资格。